pip install langchain_community langchain_groq langchain_huggingface python-dotenv requests jq wikitextparser faiss-cpu 

---
# CHUNK

In [1]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader

folder_path = "./knowledge_base/processed"
loader = DirectoryLoader(
    path=folder_path, 
    glob="*.json",
    loader_cls=JSONLoader,
    loader_kwargs = {
        "jq_schema": ".content",
        "text_content": False
    }
)

docs = loader.load()

print(f"Loaded {len(docs)} files.")
print(f"Snippet of first file: {docs[0].page_content[:50]}...")

c:\MAIN_FILES\Documents\DLSU\3rd Year\2nd Term\NLP\FINAL MP\MP\kb_extraction\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 8 files.
Snippet of first file: Bosses are aggressive, resilient enemies whose def...


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150
)

chunks = splitter.split_documents(docs)

print(f"Number of chunks created: {len(chunks)}")
print()

print("--- Chunk 0 (inspect before storing!) ---")
print(chunks[0].page_content)
print()
print("--- Chunk 1 ---")
print(chunks[1].page_content)

Number of chunks created: 210

--- Chunk 0 (inspect before storing!) ---
Bosses are aggressive, resilient enemies whose defeat advances the game in some way, such as introducing a new ore for equipment of a higher tier. Each has its own particular way of spawning: some have summoning items that can be used to spawn them manually under certain conditions, while others appear after the player interacts with the environment in a certain way. Events can also have their own event bosses (or mini-bosses) that only spawn during their respective event.

Most bosses and event bosses can pass through blocks of all types, with the exception of King Slime, Queen Slime, and the Flying Dutchman.

All regular bosses besides the Lunatic Cultist have a status message upon spawning, and all regular bosses and invasion event bosses have a status message upon defeat. Each boss has a particular theme that starts playing when they spawn. Event bosses can appear multiple times in a given event and do not hav

---
# Indexing

### Embedding

In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("Embedding model loaded: all-MiniLM-L6-v2 (384-dimensional)")
print("Embedding all chunks into the vector database...")
db = FAISS.from_documents(chunks, embeddings)

print(f"Vector database created with {len(chunks)} vectors")

C:\Users\lance\AppData\Local\Temp\ipykernel_13468\3408347235.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8589.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded: all-MiniLM-L6-v2 (384-dimensional)
Embedding all chunks into the vector database...
Vector database created with 210 vectors


In [4]:
db.save_local("faiss_index")
print("FAISS index saved to 'faiss_index/'")
print()
print("To reload later (skip the embedding step entirely):")
print("  db = FAISS.load_local('faiss_index', embeddings, allow_dangerous_deserialization=True)")

FAISS index saved to 'faiss_index/'

To reload later (skip the embedding step entirely):
  db = FAISS.load_local('faiss_index', embeddings, allow_dangerous_deserialization=True)


---

### Retrieval Phase

In [5]:
retriever = db.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created (k=3)")

Retriever created (k=3)


In [6]:
test_query = "How to craft items?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Number of chunks retrieved: {len(retrieved_docs)}")
print()
for i, doc in enumerate(retrieved_docs):
    print(f"--- Retrieved Chunk {i+1} ---")
    print(doc.page_content)
    print()

Query: How to craft items?
Number of chunks retrieved: 3

--- Retrieved Chunk 1 ---
To navigate this menu, you can click or use the mouse wheel to scroll through items. You can also see all craftable items at once by clicking the hammer icon below the word "Crafting". To select an item (as depicted by a yellow border), simply click on it, and to craft it, click it again. The newly crafted item will attach itself to your cursor, and from there it can be placed in your inventory or dropped by right-clicking outside of your inventory space. Large numbers of stackable items can be crafted by clicking and holding the icon, which quickly creates and stacks that item until the stack is full, you run out of materials, or simply let go. To exit your inventory, press the  key again.

Upon crafting, there is a 75% chance that a weapon or accessory is provided with a random modifier, slightly altering its quality. There is no way to predict what modifier will be applied on an item, and many items 

---
# Generation

pip install langchain-groq python-dotenv

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
retriever = db.as_retriever(search_kwargs={"k": 18})

print("Retriever loaded from faiss_index/")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7306.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retriever loaded from faiss_index/


In [17]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model='llama-3.1-8b-instant',
    api_key=GROQ_API_KEY
)

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# Covers: KB lookup, class info, frustrated tone, off-topic,
#         tone-deaf failure (v2), Hardmode leakage (v2),
#         over-refusal (v2), multi-part question (v3)
test_queries = [
    # 'How do I craft a Work Bench?',
    # 'What class should I pick?',
    # 'ugh i keep dying to the Eye of Cthulhu',
    # 'Is Minecraft better than Terraria?',
    # 'ugh why do i keep dying to skeletons',
    # 'What ore should I mine?',
    # 'Is the Crimson dangerous?',
    # 'What class should I pick and how do I beat Skeletron?',
    'How do I spawn wall of flesh?'
]

print('LLM loaded: llama-3.1-8b-instant (Groq)')

LLM loaded: llama-3.1-8b-instant (Groq)


---
## Prompt v1 — Baseline
Bare prompt with no rules. LLM answers freely, ignoring KB boundaries.

**Known issues:** hallucinated items, no KB grounding, off-topic answered, inconsistent length.

In [9]:
v1_prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'You are a Terraria assistant. Use the context below to answer the question.\n\n'
        'Context: {retrieved_chunks}'
    )),
    ('human', '{question}')
])

v1_chain = (
    {'retrieved_chunks': retriever | format_docs, 'question': RunnablePassthrough()}
    | v1_prompt
    | llm
    | StrOutputParser()
)

print('=== Prompt v1 Results ===\n')
for query in test_queries:
    print(f'Q: {query}')
    print(f'A: {v1_chain.invoke(query)}')
    print()

=== Prompt v1 Results ===

Q: How do I craft a Work Bench?
A: To craft a Work Bench, you will need 10 Wood. 

1. Open your inventory and navigate to the crafting menu.
2. Press the corresponding key for the crafting menu on your platform (usually  () /  () /  ()).
3. Switch to the crafting menu by pressing  /  .
4. In the crafting menu, locate the Work Bench recipe and select it.
5. If you have the required 10 Wood in your inventory, it will be listed as a possible craft.
6. Click on the Work Bench recipe to start crafting it.

Once you've crafted the Work Bench, you can place it down and use it to craft more items.

Q: What class should I pick?
A: Since you're just starting out, it's best to choose a class based on your playstyle and the type of damage you want to deal. Here are a few options:

1. **Melee:** If you enjoy close-range combat and want to deal high damage to enemies, Melee might be the way to go. Melee players typically use swords, axes, and other close-range weapons to t

---
## Prompt v2 — KB Constraint + Length Rule (Iteration 1)
Added KB-only rule, 120-word limit, structured format, and off-topic refusal.

**Fixed:** hallucinations, off-topic, inconsistent length.  
**Still broken:** tone-deaf on frustrated players, Hardmode content leakage, over-refusal on indirect questions.

In [10]:
v2_system = (
    'You are TerrariaBot, an assistant for Terraria 1.4.4.\n\n'
    'Rules:\n'
    '1. Answer ONLY using information in the Context below.\n'
    '2. If the answer is not in the Context, say: "I don\'t have that info — check terraria.wiki.gg."\n'
    '3. Lead with a direct answer. Add detail after. Use bullet points for 3+ item lists.\n'
    '4. Keep responses under 120 words unless listing steps or items.\n'
    '5. Do not answer questions unrelated to Terraria.\n\n'
    'Context: {retrieved_chunks}'
)

v2_prompt = ChatPromptTemplate.from_messages([
    ('system', v2_system),
    ('human', '{question}')
])

v2_chain = (
    {'retrieved_chunks': retriever | format_docs, 'question': RunnablePassthrough()}
    | v2_prompt
    | llm
    | StrOutputParser()
)

print('=== Prompt v2 Results ===\n')
for query in test_queries:
    print(f'Q: {query}')
    print(f'A: {v2_chain.invoke(query)}')
    print()

=== Prompt v2 Results ===

Q: How do I craft a Work Bench?
A: Craft a Work Bench by placing 10 Wood in the crafting menu.

Q: What class should I pick?
A: There isn't one class to pick; Terraria is more flexible, allowing you to experiment with different classes. However, here are the main classes and their focus:

* Melee: Focus on close-range combat and dealing high damage.
* Ranged: Focus on ranged attacks and dealing damage from a distance.
* Magic: Focus on casting spells and dealing magical damage.

Consider your playstyle and the resources available to you.

Q: ugh i keep dying to the Eye of Cthulhu
A: The Eye of Cthulhu can be challenging, especially for beginners. 

You might want to try the following:

• Upgrade your armor to provide more defense.
• Use a decent sword or staff that can deal significant damage.
• Make sure you have enough health-restoring items, such as Healing Potions or Health-restoring Potions.
• Consider using a shield to block the Eye's charges.
• If you'

---
## Prompt v3 — Few-Shot + Tone Matching + Edge Cases (Iteration 2)
Added few-shot examples, tone matching, and explicit edge case handling.

**Fixed:** frustrated tone, off-topic, indirect question over-refusal.  
**Still broken:** verbatim quoting of long chunks, multi-part questions get uneven answers.

In [19]:
v3_system = (
    'You are TerrariaBot, a friendly assistant for Terraria 1.4.4.\n\n'
    'Rules:\n'
    '1. Answer ONLY using information in the Context below.\n'
    '2. If the answer is missing, say: "I don\'t have that detail — terraria.wiki.gg is a great next step."\n'
    '3. Lead with a direct answer. Use bullet points for 3+ item lists.\n'
    '4. Match the player\'s tone: be brief and encouraging for frustrated players; be thorough for detailed questions.\n'
    '5. For off-topic questions reply: "I only know Terraria — ask me about crafting, bosses, classes, or survival!"\n'
    '6. Never reference items or mechanics absent from the provided Context.\n\n'
    'Examples:\n\n'
    'User: How do I craft a Work Bench?\n'
    'Assistant: Craft a Work Bench with 10 Wood — no crafting station needed, just open your inventory. '
    'Place it on flat ground to unlock recipes for furniture, weapons, and walls.\n\n'
    'User: What weapons are good for a ranged build early on?\n'
    'Assistant: Good early ranged options include:\n'
    '- Wooden Bow + Flaming Arrows — reliable, also lights up caves\n'
    '- Minishark — fast fire rate, bought from the Arms Dealer\n'
    '- Boomstick — high damage, found in Jungle Chests\n'
    'Pair these with an Archery Potion for extra damage.\n\n'
    'User: ugh i keep dying to the Eye of Cthulhu this is so hard\n'
    'Assistant: Hang in there! Make sure you have at least 200 HP and a flat arena with platforms before the fight. '
    'Keep moving and dodge its dash in the second phase — a bow or shuriken works well for both phases.\n\n'
    'User: Is Fortnite better than Terraria?\n'
    'Assistant: I only know Terraria — ask me about crafting, bosses, classes, or survival!\n\n'
    'Context: {retrieved_chunks}'
)

v3_prompt = ChatPromptTemplate.from_messages([
    ('system', v3_system),
    ('human', '{question}')
])

v3_chain = (
    {'retrieved_chunks': retriever | format_docs, 'question': RunnablePassthrough()}
    | v3_prompt
    | llm
    | StrOutputParser()
)

print('=== Prompt v3 Results ===\n')
for query in test_queries:
    print(f'Q: {query}')
    print(f'A: {v3_chain.invoke(query)}')
    print()

=== Prompt v3 Results ===

Q: How do I spawn wall of flesh?
A: To spawn the Wall of Flesh, you'll need to head to the underworld and fight it. The Wall of Flesh can be summoned by killing the Eye of Cthulhu in the Corruption biome. Once you've defeated the Eye, the Wall of Flesh will spawn and can be defeated to progress to Hardmode.

